# 04. Kỹ Thuật Trích Xuất Đặc Trưng Học Máy & Lát Cắt Thời Gian Cảnh Báo Sớm
## Tuần 5: Feature Engineering Cơ Bản, Snapshot 25%/50%/75%/100% & Kiểm Soát Rò Rỉ Dữ liệu
**Dự án**: LMS Tiên tiến và Learning Analytics (Bộ dữ liệu OULAD)  
**Tác giả**: Vũ Hoà Bình (MSV: 202416137)  
**Thời gian thực hiện**: Tuần 5 - Tháng 11/2026  

---
### Mục tiêu Kỹ thuật Toàn diện Tuần 5:
1. **T5.1 Click-based Features (Behavioural)**: Trích xuất `total_clicks`, `active_days`, `clicks_per_day`, `daily_click_mean`, `click_std`, `click_cv`, `active_days_ratio`, `pre_course_clicks`, `active_span_days`.
2. **T5.2 Activity-based Features (SEDL Proxies)**: Phân rã tương tác trên 20 loại `activity_type` (bao gồm `clicks_glossary`), trích xuất độ đa dạng tài nguyên `activity_diversity` và độ hỗn loạn Shannon `activity_entropy`.
3. **T5.3 Assessment-based Features**: Điểm trung bình có trọng số `weighted_avg_score`, điểm tích lũy `earned_weight_score`, điểm bài đầu tiên `first_score`, `tma1_score`, và tỷ lệ nộp bài `submission_rate` (loại trừ hoàn toàn `Exam` khỏi feature đầu vào để chống rò rỉ nhãn).
4. **T5.4 Demographics & Context**: Tải tín chỉ `studied_credits`, số lần học lại `num_of_prev_attempts`, mã hóa học vấn, kinh tế vùng `imd_band`, độ tuổi.
5. **Snapshot Temporal Modeling**: Trích xuất 4 lát cắt thời gian $T_{\text{cutoff}} \in \{25\%, 50\%, 75\%, 100\%\}$ đảm bảo Zero Temporal Leakage tuyệt đối.
6. **Cohort Dynamic Modeling**: Đối sánh Full Cohort ($N=32,593$), Pre-course Active Cohort ($N=29,915$) và Dynamic Active Cohort tại từng mốc thời gian.
7. **Feature Importance & Xếp hạng Tầm quan trọng**: Tính toán Point-Biserial $r$, Spearman $\rho$, Cohen's $d$, Mutual Information và Random Forest Gini/Permutation Importance cho 76 đặc trưng.
8. **Thực nghiệm Benchmark**: Huấn luyện đối sánh 92 cấu hình mô hình qua 4 mốc thời gian và 2 chiến lược chia tập (Stratified Random vs Group-Stratified).

In [1]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn thư mục chuẩn
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'

sys.path.insert(0, str(PROJECT_ROOT))
from src.feature_engineering import OULADFeatureEngineer, build_feature_catalog

print(f'Project root: {PROJECT_ROOT}')
print(f'Processed data dir: {PROCESSED_DIR}')

## 1. Nạp và Kiểm tra Danh mục Đặc trưng (Feature Catalog 76 Features)
Danh mục tổng hợp toàn bộ 76 đặc trưng được phân loại theo 4 nhóm kỹ thuật và khung lý thuyết SEDL (Bergdahl et al., 2024).

In [2]:
catalog_path = PROCESSED_DIR / 'feature_catalog_week5.csv'
df_catalog = pd.read_csv(catalog_path)
print(f'Tổng số đặc trưng đã thiết kế: {len(df_catalog)}')
print('\nPhân bổ theo nhóm đặc trưng:')
print(df_catalog['feature_group'].value_counts())
print('\nPhân bổ theo chiều SEDL:')
print(df_catalog['sedl_dimension'].value_counts())
df_catalog[['feature_name', 'feature_group', 'sedl_dimension', 'cohens_d_100', 'mutual_info_100', 'rf_gini_importance_100']].head(10)

## 2. Kiểm tra Tiến trình Đặc trưng qua 4 Lát cắt Thời gian (Snapshots 25%, 50%, 75%, 100%)
Quan sát sự tích lũy thông tin không rò rỉ tương lai từ mốc cảnh báo sớm đến cuối khóa.

In [3]:
metrics_path = PROCESSED_DIR / 'week5_experiment_metrics.json'
with open(metrics_path, 'r', encoding='utf-8') as f:
    metrics_data = json.load(f)

progression = metrics_data['snapshot_progression_stats']
prog_df = pd.DataFrame(progression).T
print('Thống kê biến thiên qua 4 lát cắt thời gian:')
prog_df[['full_cohort_size', 'dynamic_active_size', 'dynamic_active_at_risk_rate', 'mean_total_clicks_not_at_risk', 'mean_total_clicks_at_risk', 'mean_earned_weight_not_at_risk', 'mean_earned_weight_at_risk']]

## 3. Trực quan hóa Xếp hạng Tầm quan trọng Đặc trưng (Top Features Ranking)
Hiển thị Top 10 đặc trưng phân biệt mạnh nhất ở mốc Cảnh báo sớm 25% và Toàn khóa 100%.

In [4]:
top_100 = pd.DataFrame(metrics_data['importance_summary']['top20_by_rf_100pct'])
top_25 = pd.DataFrame(metrics_data['importance_summary']['top20_by_rf_25pct'])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sns.barplot(data=top_25.head(10), x='rf_gini_importance_25', y='feature_name', ax=axes[0], palette='Blues_r', edgecolor='black')
axes[0].set_title('Top 10 Đặc trưng Quan trọng nhất tại Mốc 25% (Cảnh báo sớm)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Random Forest Gini Importance (25%)', fontsize=11)

sns.barplot(data=top_100.head(10), x='rf_gini_importance_100', y='feature_name', ax=axes[1], palette='Oranges_r', edgecolor='black')
axes[1].set_title('Top 10 Đặc trưng Quan trọng nhất tại Mốc 100% (Toàn khóa)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Random Forest Gini Importance (100%)', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Đối sánh Hiệu năng Mô hình Dự đoán: Bộ Đặc trưng Tuần 5 vs Tuần 3-4
Đối chiếu định lượng trên tập kiểm thử Test độc lập dưới chiến lược Group-Stratified Split.

In [5]:
experiments = pd.DataFrame(metrics_data['experiments'])
filtered_exp = experiments[
    (experiments['cohort'] == 'full') & 
    (experiments['split_strategy'] == 'group_stratified') & 
    (experiments['model'] == 'Random Forest')
]
print('Đối sánh Random Forest (Group-Stratified Split, Full Cohort):')
filtered_exp[['snapshot', 'feature_set', 'num_features_encoded', 'test_acc', 'test_prec', 'test_rec', 'test_f1', 'test_auc']]

## 5. Mở rộng Tuần 6: Đặc trưng Nâng cao (Temporal, Regularity, Trajectory) & Hệ thống Cảnh báo Sớm
Bổ sung 16 đặc trưng Tuần 6 (danh mục mở rộng 76 -> 92 đặc trưng; ma trận mã hóa 98 -> 114 biến) với 100% Zero Temporal Leakage và tinh chỉnh ngưỡng trên tập Validation (`LightGBM (Balanced)`).

In [6]:
catalog_w6 = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'feature_catalog_week6.csv')
with open(PROJECT_ROOT / 'data' / 'processed' / 'week6_experiment_metrics.json', 'r', encoding='utf-8') as f:
    w6_metrics = json.load(f)

print(f'Tổng số đặc trưng Danh mục Tuần 6: {len(catalog_w6)} (trong đó {int(catalog_w6["is_week6_new"].sum())} đặc trưng mới Tuần 6)')
top15_w6 = catalog_w6.sort_values('rf_gini_importance_25', ascending=False).head(15)
display(top15_w6[['feature_name', 'feature_group', 'is_week6_new', 'point_biserial_r_25', 'cohens_d_25', 'rf_gini_importance_25', 'rf_gini_importance_100']])

ew_df = pd.DataFrame(w6_metrics['early_warning_tuned'])
print('\nHiệu năng Early Warning (Dynamic Active Cohort, LightGBM Balanced + Val-Tuned Threshold):')
display(ew_df[(ew_df['cohort'] == 'dynamic_active') & (ew_df['model'] == 'LightGBM (Balanced)')][[
    'snapshot', 'n_cohort', 'default_05_test_recall', 'tuned_threshold_val', 'test_recall', 'test_precision', 'test_f1', 'test_roc_auc', 'test_pr_auc'
]])

In [7]:
w6_exp = pd.DataFrame(w6_metrics['experiments'])
dyn_lgb = w6_exp[(w6_exp['cohort'] == 'dynamic_active') & (w6_exp['model'] == 'LightGBM')]
print('Đối sánh LightGBM Chuẩn (week5_expanded vs week6_advanced) trên Dynamic Active Cohort:')
display(dyn_lgb[['snapshot', 'feature_set', 'n_features_encoded', 'test_roc_auc', 'test_f1', 'test_recall', 'test_precision']])